In [18]:
!pip install requests beautifulsoup4 pandas lxml

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\mahar\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [22]:
from bs4 import BeautifulSoup
import pandas as pd
import time
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
import undetected_chromedriver as uc
import DrissionPage as dp
from DrissionPage import ChromiumPage

In [33]:
from bs4 import BeautifulSoup
import pandas as pd
import os
import re

def parse_clean_local_html():
    file_path = "tpu_clean.html" 
    
    if not os.path.exists(file_path):
        print(f"Error: '{file_path}' not found.")
        return None

    print(f"Parsing local file {file_path} for GPU data and Images...")
    
    with open(file_path, "r", encoding="utf-8") as f:
        html_content = f.read()

    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Target the correct table class found in the HTML
    table = soup.find('table', class_='items-desktop-table')
    
    if not table:
        print("Could not locate the 'items-desktop-table'. The HTML might not contain the data.")
        return None

    data = []
    tbody = table.find('tbody')
    rows = tbody.find_all('tr') if tbody else table.find_all('tr')
    
    for row in rows:
        cells = row.find_all(['td', 'th'])
        
        # Skip internal headers or weird spacer rows
        if not cells or cells[0].name == 'th' or 'vendor-' not in row.get('class', ['vendor-'])[0]:
            continue
            
        row_data = {
            'Name': None,
            'Image_URL': None,
            'Bus': None,
            'Memory': None,
            'GPU Clock': None,
            'Memory Clock': None,
            'Cores / TMUs / ROPs': None
        }
        
        try:
            # 1. Product Name (usually in a link in the first column)
            name_cell = cells[0]
            name_link = name_cell.find('a')
            if name_link:
                row_data['Name'] = " ".join(name_link.text.split())
            else:
                row_data['Name'] = " ".join(name_cell.get_text().split())

            # 2. Image URL 
            img_tag = row.find('img')
            if img_tag and 'src' in img_tag.attrs:
                img_url = img_tag['src']
                if 'data-src' in img_tag.attrs:
                    img_url = img_tag['data-src']
                    
                if img_url.startswith('//'):
                    img_url = f"https:{img_url}"
                elif img_url.startswith('/'):
                    img_url = f"https://www.techpowerup.com{img_url}"
                elif not img_url.startswith('http'):
                    filename = os.path.basename(img_url).split('?')[0] 
                    img_url = f"https://tpucdn.com/gpu-specs/images-new/c/{filename}"
                
                row_data['Image_URL'] = img_url

            # 3. Spec Columns
            # Clean up the text of all cells
            cell_texts = [re.sub(r'\s+', ' ', cell.text.strip()) for cell in cells if cell.text.strip()]
            
            # The columns align from the right side reliably
            if len(cell_texts) >= 5:
                row_data['Cores / TMUs / ROPs'] = cell_texts[-1]
                row_data['Memory Clock'] = cell_texts[-2]
                row_data['GPU Clock'] = cell_texts[-3]
                row_data['Memory'] = cell_texts[-4]
                row_data['Bus'] = cell_texts[-5]

            data.append(row_data)

        except Exception as e:
            continue

    df = pd.DataFrame(data)
    return df

# Execute
gpu_dataframe = parse_clean_local_html()

if gpu_dataframe is not None:
    # Cleanup dataset
    gpu_dataframe.replace(['Unknown', '', 'N/A', 'None'], pd.NA, inplace=True)
    gpu_dataframe.dropna(subset=['Name'], inplace=True)
    
    display(gpu_dataframe.head())
    
    csv_filename = "techpowerup_gpus_with_images.csv"
    gpu_dataframe.to_csv(csv_filename, index=False)
    print(f"\nData successfully saved to {csv_filename} ({len(gpu_dataframe)} rows extracted)")

Parsing local file tpu_clean.html for GPU data and Images...


,Name,Image_URL,Bus,Memory,GPU Clock,Memory Clock,Cores / TMUs / ROPs
0,Radeon RX 9070 XT,https://tpucdn.com/gpu-specs/images-new/c/4229...,PCIe 5.0 x16,16 GB / GDDR6 / 256 bit,2970 MHz,2518 MHz,4096 / 256 / 128
1,GeForce RTX 5090,https://tpucdn.com/gpu-specs/images-new/c/4216...,PCIe 5.0 x16,32 GB / GDDR7 / 512 bit,2407 MHz,1750 MHz,21760 / 680 / 176
2,GeForce RTX 5070,https://tpucdn.com/gpu-specs/images-new/c/4218...,PCIe 5.0 x16,12 GB / GDDR7 / 192 bit,2512 MHz,1750 MHz,6144 / 192 / 80
3,Radeon RX 9060 XT 16 GB,https://tpucdn.com/gpu-specs/images-new/c/4293...,PCIe 5.0 x16,16 GB / GDDR6 / 128 bit,3130 MHz,2518 MHz,2048 / 128 / 64
4,GeForce RTX 5070 Ti,https://tpucdn.com/gpu-specs/images-new/c/4243...,PCIe 5.0 x16,16 GB / GDDR7 / 256 bit,2452 MHz,1750 MHz,8960 / 280 / 96



Data successfully saved to techpowerup_gpus_with_images.csv (100 rows extracted)


In [ ]:
import pandas as pd
from IPython.display import HTML

loaded_df = pd.read_csv("techpowerup_gpus_with_images.csv")

# Function to convert image URLs into HTML image tags
def to_img_tag(url):
    if pd.isna(url):
        return ""
    return f'<img src="{url}" width="100" />'

# Apply the formatting and display
html_table = loaded_df.head(100).to_html(escape=False, formatters=dict(Image_URL=to_img_tag))
display(HTML(html_table))

,Name,Image_URL,Bus,Memory,GPU Clock,Memory Clock,Cores / TMUs / ROPs
0,Radeon RX 9070 XT,,PCIe 5.0 x16,16 GB / GDDR6 / 256 bit,2970 MHz,2518 MHz,4096 / 256 / 128
1,GeForce RTX 5090,,PCIe 5.0 x16,32 GB / GDDR7 / 512 bit,2407 MHz,1750 MHz,21760 / 680 / 176
2,GeForce RTX 5070,,PCIe 5.0 x16,12 GB / GDDR7 / 192 bit,2512 MHz,1750 MHz,6144 / 192 / 80
3,Radeon RX 9060 XT 16 GB,,PCIe 5.0 x16,16 GB / GDDR6 / 128 bit,3130 MHz,2518 MHz,2048 / 128 / 64
4,GeForce RTX 5070 Ti,,PCIe 5.0 x16,16 GB / GDDR7 / 256 bit,2452 MHz,1750 MHz,8960 / 280 / 96
5,GeForce RTX 5060,,PCIe 5.0 x8,8 GB / GDDR7 / 128 bit,2497 MHz,1750 MHz,3840 / 120 / 48
6,GeForce RTX 5060 Ti 16 GB,,PCIe 5.0 x8,16 GB / GDDR7 / 128 bit,2572 MHz,1750 MHz,4608 / 144 / 48
7,GeForce RTX 5080,,PCIe 5.0 x16,16 GB / GDDR7 / 256 bit,2617 MHz,1875 MHz,10752 / 336 / 112
8,GeForce RTX 3060 12 GB,,PCIe 4.0 x16,12 GB / GDDR6 / 192 bit,1777 MHz,1875 MHz,3584 / 112 / 48
9,GeForce RTX 3080,,PCIe 4.0 x16,10 GB / GDDR6X / 320 bit,1710 MHz,1188 MHz,8704 / 272 / 96
